In [ ]:
import pandas as pd

def get_tat_times(
    df,
    id_col = "lab_test_id",
    collect_col = "latest_collect_dt",
    received_col = "latest_received_dt",
    result_col = "latest_result_dt",
    cap_q = 0.99,
    verbose: True,
):
    df = df.copy()

    # de-dupe (keep first occurrence)
    if id_col in df.columns:
        df = df.drop_duplicates(subset=[id_col])

    # parse datetimes safely
    for c in [collect_col, received_col, result_col]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    # compute durations in hours
    df["tat_hours"] = (df[result_col] - df[collect_col]).dt.total_seconds() / 3600
    df["received_hours"] = (df[received_col] - df[collect_col]).dt.total_seconds() / 3600
    df["processed_hours"] = (df[result_col] - df[received_col]).dt.total_seconds() / 3600

    new_cols = ["tat_hours", "received_hours", "processed_hours"]

    # clean: keep non-missing + non-negative for all three
    df = df.dropna(subset=new_cols)
    df = df[(df[new_cols] >= 0).all(axis=1)]

    # stats
    stats = df[new_cols].agg(["count", "mean", "median", lambda s: s.quantile(cap_q)]).T
    stats.columns = ["n", "mean_hours", "median_hours", f"cap_p{int(cap_q*100)}_hours"]
    stats["mean_days"] = stats["mean_hours"] / 24
    stats["median_days"] = stats["median_hours"] / 24
    stats[f"cap_p{int(cap_q*100)}_days"] = stats[f"cap_p{int(cap_q*100)}_hours"] / 24

    if verbose:
        print(f"N rows after cleaning: {len(df):,}\n")
        for metric, row in stats.iterrows():
            cap_h = row[f"cap_p{int(cap_q*100)}_hours"]
            cap_d = row[f"cap_p{int(cap_q*100)}_days"]
            print(
                f"{metric}: n={int(row['n']):,} | "
                f"mean={row['mean_hours']:.2f}h ({row['mean_days']:.2f}d) | "
                f"median={row['median_hours']:.2f}h ({row['median_days']:.2f}d) | "
                f"{int(cap_q*100)}th% cap={cap_h:.2f}h ({cap_d:.2f}d)"
            )

    return df, stats
